In [1]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"GPU count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.9.1+cu130
CUDA available: True
CUDA version: 13.0
GPU count: 1
GPU name: NVIDIA GeForce RTX 5090


In [ ]:
import sys
print(sys.executable)

/home/zcorn/anaconda3/envs/py314-llm/bin/python


# WRI1 (WRINKLED1) Binding Affinity Benchmark

Evaluating trained Protein-DNA binding models on the WRI1 transcription factor binding affinity dataset (MST measurements from Payton).

**Dataset:**
- AtWRI1 (WRINKLED1) transcription factor from Arabidopsis thaliana
- DNA sequences: synthetic sequences with 18bp AW-boxes
- Target: Kd (nM) binding affinity values

**Models to evaluate:**
- DNA-only baseline
- Protein-only baseline  
- Concatenation
- Contrastive
- Attention
- Composition

In [3]:
import torch
import numpy as np
import pandas as pd
from torch import nn
from tqdm import tqdm
import torch.nn.functional as F
from torch.nn import CosineSimilarity
from sklearn.metrics import (
    roc_auc_score, average_precision_score, 
    matthews_corrcoef, accuracy_score,
    mean_squared_error, mean_absolute_error,
    r2_score
)
from scipy.stats import spearmanr, pearsonr
from transformers import AutoModelForMaskedLM, AutoTokenizer, AutoModel, BertConfig
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Embedding dimensions (must match training)
DNA_EMB_DIM = 768   # DNABERT-2 output dimension
PROT_EMB_DIM = 320  # ESM2 output dimension

# Paths
RESULTS_DIR = "/home/zcorn/Projects/BioLLMComposition/results"
DATA_PATH = "/home/zcorn/Projects/proteinDNA_data/processed/AtWRI1_binding_affinities_towards_AW-boxes_processed_20260130.csv"

/home/zcorn/anaconda3/envs/py314-llm/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


## 1. Load Dataset and WRI1 Protein Sequence

In [4]:
# Load dataset
df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Dataset shape: (176, 4)
Columns: ['synthetic_seq', 'aw_18bp', 'kd_nm_average', 'kd_nm_std_dev']


,synthetic_seq,aw_18bp,kd_nm_average,kd_nm_std_dev
0,TCTCCGCCTTCGTAAGTTCCGCCGAAAA,GCCTTCGTAAGTTCCGCC,2.47,1.42
1,TCTTCGCTTCGGTACTTCCGAAGATTTCA,CGCTTCGGTACTTCCGAA,45.26,3.76
2,AAAACGACGATCTCTTCGATGCTTTCTC,AGCATCGAAGAGATCGTC,70.19,9.42
3,TCTCACACTTCGATATCTCCGCCTTCATT,CACTTCGATATCTCCGCC,0.56,0.28
4,TTGACTCCTTAGTTTTCATCGCCTCCGCC,TCCTTAGTTTTCATCGCC,10.31,6.68


In [5]:
# Check Kd values - some are "not binding", "no binding", "non-binding"
print("Unique non-numeric Kd values:")
non_numeric = df[pd.to_numeric(df['kd_nm_average'], errors='coerce').isna()]['kd_nm_average'].unique()
print(non_numeric)
print(f"\nCount of non-numeric: {len(df[pd.to_numeric(df['kd_nm_average'], errors='coerce').isna()])}")

# Create binary labels: binding (1) vs non-binding (0)
df['kd_numeric'] = pd.to_numeric(df['kd_nm_average'], errors='coerce')
df['is_binding'] = df['kd_numeric'].notna().astype(int)

# For binders, also create log10(Kd) for regression
df['log_kd'] = np.log10(df['kd_numeric'].replace(0, np.nan))

print(f"\nBinding: {df['is_binding'].sum()}, Non-binding: {(df['is_binding'] == 0).sum()}")
print(f"\nKd distribution for binders:")
print(df[df['is_binding'] == 1]['kd_numeric'].describe())

Unique non-numeric Kd values:
['not binding' 'no binding' 'non-binding']

Count of non-numeric: 35

Binding: 141, Non-binding: 35

Kd distribution for binders:
count     141.000000
mean      103.449787
std       206.739848
min         0.000000
25%         2.010000
50%        12.200000
75%        80.100000
max      1099.540000
Name: kd_numeric, dtype: float64


In [6]:
# WRI1 (WRINKLED1) protein sequence from Arabidopsis thaliana
# UniProt: Q9SRF7
# This is the AP2/ERF transcription factor that regulates fatty acid synthesis

WRI1_PROTEIN_SEQ = """MDLSDSPFNFLNSGVNSLHTEKHMPSSDDDDDDHQHHFDHHTQHLHLSSPSEPKTPPVCS
SGYITAPYSKVNLRTYRGVTRHQRTWGKWAAEIRDPAKNGARVWLGTYDTAEEAALAYDQ
AAFQLRGPSAHLNFPALRLYENFSYRGISRGSRTTKQLVDSHLANLFPTPSHSNASSSDI
FSNPSSASDTSISFSQNNLLSSNGSNHGSSVNNSTVVNADSGSLGYNSSIVQPTISTNYS
TYHSTNSIPFMQPLGEPKRKLTTVRYRGVARHHVDWGKWVSEVREPNLGTRIWLGTYDTS
EEIAAYDKAALLFRQRSTFKVYTDEDYSYKKQPLLHQGLSSSPLMFPPNPFNVASTAAKS
STDASTSSSCSFLSPFASNGPVYGSAPGNAYSNVNIVNVNLPGVIGSQGLSYTTTMPGVN
GQTTTWSPLPFNSSNVNGSTTMGLSLNGQNSLAPISFDPNQQLG""".replace('\n', '')

print(f"WRI1 protein length: {len(WRI1_PROTEIN_SEQ)} amino acids")
print(f"First 50 aa: {WRI1_PROTEIN_SEQ[:50]}...")

WRI1 protein length: 464 amino acids
First 50 aa: MDLSDSPFNFLNSGVNSLHTEKHMPSSDDDDDDHQHHFDHHTQHLHLSSP...


In [7]:
df

,synthetic_seq,aw_18bp,kd_nm_average,kd_nm_std_dev,kd_numeric,is_binding,log_kd
0,TCTCCGCCTTCGTAAGTTCCGCCGAAAA,GCCTTCGTAAGTTCCGCC,2.47,1.42,2.47,1,0.392697
1,TCTTCGCTTCGGTACTTCCGAAGATTTCA,CGCTTCGGTACTTCCGAA,45.26,3.76,45.26,1,1.655715
2,AAAACGACGATCTCTTCGATGCTTTCTC,AGCATCGAAGAGATCGTC,70.19,9.42,70.19,1,1.846275
3,TCTCACACTTCGATATCTCCGCCTTCATT,CACTTCGATATCTCCGCC,0.56,0.28,0.56,1,-0.251812
4,TTGACTCCTTAGTTTTCATCGCCTCCGCC,TCCTTAGTTTTCATCGCC,10.31,6.68,10.31,1,1.013259
...,...,...,...,...,...,...,...
171,AAACCCACCTCGAGACCATCATCATCTT,CACCTCGAGACCATCATC,792.58,NaN,792.58,1,2.899043
172,TCGTGGACGTGGTTCTTCTCGCCTTATC,GACGTGGTTCTTCTCGCC,no binding,NaN,NaN,0,NaN
173,AGGCACTCTTTGTCCATCTCGTTGGTAT,CTCTTTGTCCATCTCGTT,no binding,NaN,NaN,0,NaN
174,GATCAAGCTTTGAATAAGACGTGTCAAA,AGCTTTGAATAAGACGTG,no binding,NaN,NaN,0,NaN


## 2. Load Language Models (DNABERT-2 and ESM2)

In [8]:
print("Loading DNABERT-2 model...")
dna_tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)
dna_config = BertConfig.from_pretrained("zhihan1996/DNABERT-2-117M")
dna_lm = AutoModel.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True, config=dna_config)
dna_lm = dna_lm.to(device).eval()

print("Loading ESM2 model...")
esm_layers = 6
esm_params = 8
prot_lm = AutoModelForMaskedLM.from_pretrained(f'facebook/esm2_t{esm_layers}_{esm_params}M_UR50D').to(device).eval()
prot_tokenizer = AutoTokenizer.from_pretrained(f'facebook/esm2_t{esm_layers}_{esm_params}M_UR50D')

print("Models loaded successfully!")

Loading DNABERT-2 model...


Some weights of BertModel were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading ESM2 model...
Models loaded successfully!


In [9]:
# Embedding extraction functions

def get_dna_mean_rep(sequence):
    """Get mean pooled embedding from DNABERT-2."""
    inputs = dna_tokenizer(sequence, return_tensors='pt', truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = dna_lm(**inputs)
    if isinstance(outputs, tuple):
        hidden_states = outputs[0][0]
    else:
        hidden_states = outputs.last_hidden_state[0]
    mean_embedding = hidden_states.mean(dim=0)
    return mean_embedding.cpu().numpy()

def get_prot_mean_rep(sequence):
    """Get mean pooled embedding from ESM2."""
    token_ids = prot_tokenizer(sequence, return_tensors='pt', truncation=True, max_length=1024).to(device)
    with torch.no_grad():
        results = prot_lm.forward(token_ids.input_ids, output_hidden_states=True)
    representations = results.hidden_states[-1][0]
    mean_embedding = representations[1:min(len(sequence)+1, representations.shape[0]-1)].mean(dim=0)
    return mean_embedding.cpu().numpy()

print("Embedding functions defined.")

Embedding functions defined.


## 3. Define Model Architectures (matching training)

In [10]:
# Model architectures (must match training)

class CLModel(nn.Module):
    """Contrastive Learning Model"""
    def __init__(self):
        super(CLModel, self).__init__()
        self.dna_proj = torch.nn.Sequential(
            torch.nn.Linear(DNA_EMB_DIM, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 128),
        )
        self.protein_proj = torch.nn.Sequential(
            torch.nn.Linear(PROT_EMB_DIM, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 128),
        )

    def forward(self, dna, prot):
        dna = self.dna_proj(dna)
        prot = self.protein_proj(prot)
        dna = F.normalize(dna, p=2, dim=1)
        prot = F.normalize(prot, p=2, dim=1)
        return dna, prot


# Target layers for cross-attention in ESM2
target_layers = [0, 3, 5]


class AttentionModel(nn.Module):
    """Cross-attention model: Protein attends to DNA."""
    def __init__(self, dna_model, prot_model, dna_tok, prot_tok):
        super(AttentionModel, self).__init__()
        self.dna_model = dna_model
        self.prot_model = prot_model
        self.dna_tokenizer = dna_tok
        self.prot_tokenizer = prot_tok
        
        self.dna_proj = nn.Linear(DNA_EMB_DIM, PROT_EMB_DIM)
        self.attn = nn.MultiheadAttention(PROT_EMB_DIM, 1, batch_first=True)
        self.prediction_head = torch.nn.Sequential(
            torch.nn.Linear(PROT_EMB_DIM, 2),
        )
        self.post_attn_norm = nn.LayerNorm(PROT_EMB_DIM)
        
        for param in self.dna_model.parameters():
            param.requires_grad = False
        for param in self.prot_model.parameters():
            param.requires_grad = False

    def forward(self, dna_tokens, prot_tokens, attn_mask=None):
        with torch.no_grad():
            dna_outputs = self.dna_model(
                input_ids=dna_tokens['input_ids'],
                attention_mask=dna_tokens['attention_mask']
            )
        if isinstance(dna_outputs, tuple):
            dna_hidden = dna_outputs[0]
        else:
            dna_hidden = dna_outputs.last_hidden_state
        
        dna_hidden = self.dna_proj(dna_hidden)
        
        with torch.no_grad():
            prot_outputs = self.prot_model.forward(
                prot_tokens['input_ids'],
                prot_tokens['attention_mask'],
                output_hidden_states=True
            )
        prot_hidden = prot_outputs.hidden_states[-1]
        
        attn_mask = torch.matmul(
            prot_tokens["attention_mask"].unsqueeze(2).float(),
            dna_tokens['attention_mask'].unsqueeze(1).float(),
        ).repeat(1, 1, 1)
        
        output, attn_weights = self.attn(
            query=prot_hidden, key=dna_hidden, value=dna_hidden,
            attn_mask=attn_mask, average_attn_weights=False
        )
        output = self.post_attn_norm(output) + prot_hidden
        
        mask_sum = prot_tokens['attention_mask'].sum(dim=1, keepdim=True).clamp(min=1e-6)
        output = (output * prot_tokens['attention_mask'].unsqueeze(2)).sum(dim=1) / mask_sum
        
        return self.prediction_head(output), output


class CompositionModel(nn.Module):
    """Composition of Language Models for Protein-DNA Binding."""
    def __init__(self, dna_model, prot_model, dna_tok, prot_tok):
        super(CompositionModel, self).__init__()
        self.dna_model = dna_model
        self.prot_model = prot_model
        self.dna_tokenizer = dna_tok
        self.prot_tokenizer = prot_tok
        
        self.dna_proj = nn.Linear(DNA_EMB_DIM, PROT_EMB_DIM)
        self.cross_atten_layers = nn.ModuleList([
            nn.MultiheadAttention(PROT_EMB_DIM, 20, batch_first=True) for _ in range(len(target_layers))
        ])
        self.post_attn_norms = nn.ModuleList([
            nn.LayerNorm(PROT_EMB_DIM) for _ in range(len(target_layers))
        ])
        self.prediction_head = torch.nn.Sequential(
            torch.nn.Linear(PROT_EMB_DIM, 2),
        )
        self.esm_layers = 6
        
        for param in self.dna_model.parameters():
            param.requires_grad = False
        for param in self.prot_model.parameters():
            param.requires_grad = False

    def forward(self, dna_input, prot_input, attn_mask=None):
        with torch.no_grad():
            dna_outputs = self.dna_model(
                input_ids=dna_input['input_ids'],
                attention_mask=dna_input['attention_mask']
            )
        if isinstance(dna_outputs, tuple):
            dna = dna_outputs[0]
        else:
            dna = dna_outputs.last_hidden_state
        
        dna = self.dna_proj(dna)
        
        attn_mask = torch.matmul(
            prot_input["attention_mask"].unsqueeze(2).float(),
            dna_input['attention_mask'].unsqueeze(1).float(),
        ).repeat(20, 1, 1)
        
        prot_attn_mask = self.prot_model.get_extended_attention_mask(
            prot_input['attention_mask'], prot_input["input_ids"].size()
        )
        
        prot = self.prot_model.esm.embeddings(
            prot_input["input_ids"], prot_input["attention_mask"]
        ).to(device)
        
        counter = 0
        for i in range(0, self.esm_layers):
            prot = self.prot_model.esm.encoder.layer[i](prot, prot_attn_mask)[0]
            
            if i in target_layers:
                attn_out, _ = self.cross_atten_layers[counter](
                    query=prot, key=dna, value=dna,
                    attn_mask=attn_mask, average_attn_weights=False
                )
                attn_out = self.post_attn_norms[counter](attn_out)
                prot = prot + attn_out
                counter += 1
        
        prot = self.prot_model.esm.encoder.emb_layer_norm_after(prot)
        
        mask_sum = prot_input['attention_mask'].sum(dim=1, keepdim=True).clamp(min=1e-6)
        prot = (prot * prot_input['attention_mask'].unsqueeze(2)).sum(dim=1) / mask_sum
        
        return self.prediction_head(prot), prot

print("Model classes defined.")

Model classes defined.


## 4. Extract Embeddings for WRI1 Dataset

In [11]:
# Extract embeddings for all DNA sequences
print("Extracting DNA embeddings...")
dna_seqs = df['synthetic_seq'].tolist()
dna_embs = []
for seq in tqdm(dna_seqs):
    dna_embs.append(get_dna_mean_rep(seq))
dna_embs = np.array(dna_embs)

# Get WRI1 protein embedding (same for all samples)
print("\nExtracting WRI1 protein embedding...")
wri1_emb = get_prot_mean_rep(WRI1_PROTEIN_SEQ)
# Repeat for all samples
protein_embs = np.tile(wri1_emb, (len(df), 1))

print(f"\nDNA embeddings shape: {dna_embs.shape}")
print(f"Protein embeddings shape: {protein_embs.shape}")

Extracting DNA embeddings...


100%|████████████████████████████████████████████████████████████| 176/176 [00:01<00:00, 158.65it/s]



Extracting WRI1 protein embedding...

DNA embeddings shape: (176, 768)
Protein embeddings shape: (176, 320)


In [12]:
# Tokenize sequences for attention/composition models
print("Tokenizing sequences...")

# For attention models, we need to tokenize DNA and protein sequences
max_dna_len = min(512, max(len(s) for s in dna_seqs) + 10)
max_prot_len = min(1024, len(WRI1_PROTEIN_SEQ) + 10)
print(f"Max DNA length: {max_dna_len}, Max Protein length: {max_prot_len}")

dna_tokens = dna_tokenizer(dna_seqs, return_tensors='pt', padding='max_length', 
                           max_length=max_dna_len, truncation=True)

# WRI1 protein - same for all samples
prot_tokens_single = prot_tokenizer([WRI1_PROTEIN_SEQ], return_tensors='pt', padding='max_length', 
                                     max_length=max_prot_len, truncation=True)
# Repeat for all samples
prot_tokens = {
    'input_ids': prot_tokens_single['input_ids'].repeat(len(df), 1),
    'attention_mask': prot_tokens_single['attention_mask'].repeat(len(df), 1)
}

print(f"DNA tokens shape: {dna_tokens['input_ids'].shape}")
print(f"Protein tokens shape: {prot_tokens['input_ids'].shape}")

Tokenizing sequences...
Max DNA length: 40, Max Protein length: 474
DNA tokens shape: torch.Size([176, 40])
Protein tokens shape: torch.Size([176, 474])


## 5. Load Trained Models and Evaluate

In [13]:
# Convert to tensors
dna_embs_tensor = torch.tensor(dna_embs, dtype=torch.float32)
protein_embs_tensor = torch.tensor(protein_embs, dtype=torch.float32)
labels = torch.tensor(df['is_binding'].values, dtype=torch.long)

# Move tokens to device
dna_tokens_device = {k: v.to(device) for k, v in dna_tokens.items()}
prot_tokens_device = {k: v.to(device) for k, v in prot_tokens.items()}

# Results storage
results = {}

print("="*60)
print("Evaluating models on WRI1 benchmark")
print("="*60)

Evaluating models on WRI1 benchmark


In [16]:
# 1. DNA-only model
print("\n--- DNA-only Model ---")
dna_model = torch.nn.Sequential(torch.nn.Linear(DNA_EMB_DIM, 2))
dna_model.load_state_dict(torch.load(f"{RESULTS_DIR}/dna_emb_model.pth", map_location=device))
dna_model = dna_model.to(device).eval()

with torch.no_grad():
    dna_out = dna_model(dna_embs_tensor.to(device))
    dna_probs = torch.softmax(dna_out, dim=1)[:, 1].cpu().numpy()
    dna_preds = torch.argmax(dna_out, dim=1).cpu().numpy()

results['DNA-only'] = {
    'probs': dna_probs,
    'preds': dna_preds,
    'acc': accuracy_score(labels.numpy(), dna_preds),
    'roc_auc': roc_auc_score(labels.numpy(), dna_probs),
    'pr_auc': average_precision_score(labels.numpy(), dna_probs),
    'mcc': matthews_corrcoef(labels.numpy(), dna_preds)
}
print(f"Accuracy: {results['DNA-only']['acc']:.4f}")
print(f"ROC-AUC: {results['DNA-only']['roc_auc']:.4f}")
print(f"PR-AUC: {results['DNA-only']['pr_auc']:.4f}")
print(f"MCC: {results['DNA-only']['mcc']:.4f}")


--- DNA-only Model ---
Accuracy: 0.6364
ROC-AUC: 0.4296
PR-AUC: 0.7675
MCC: -0.0514


In [18]:
# 2. Protein-only model
print("\n--- Protein-only Model ---")
prot_model = torch.nn.Sequential(torch.nn.Linear(PROT_EMB_DIM, 2))
prot_model.load_state_dict(torch.load(f"{RESULTS_DIR}/protein_emb_model.pth", map_location=device))
prot_model = prot_model.to(device).eval()

with torch.no_grad():
    prot_out = prot_model(protein_embs_tensor.to(device))
    prot_probs = torch.softmax(prot_out, dim=1)[:, 1].cpu().numpy()
    prot_preds = torch.argmax(prot_out, dim=1).cpu().numpy()

results['Protein-only'] = {
    'probs': prot_probs,
    'preds': prot_preds,
    'acc': accuracy_score(labels.numpy(), prot_preds),
    'roc_auc': roc_auc_score(labels.numpy(), prot_probs) if len(np.unique(prot_probs)) > 1 else 0.5,
    'pr_auc': average_precision_score(labels.numpy(), prot_probs) if len(np.unique(prot_probs)) > 1 else labels.numpy().mean(),
    'mcc': matthews_corrcoef(labels.numpy(), prot_preds)
}
print(f"Accuracy: {results['Protein-only']['acc']:.4f}")
print(f"ROC-AUC: {results['Protein-only']['roc_auc']:.4f}")
print(f"PR-AUC: {results['Protein-only']['pr_auc']:.4f}")
print(f"MCC: {results['Protein-only']['mcc']:.4f}")
print("Note: Same protein for all samples, so predictions may be constant.")


--- Protein-only Model ---
Accuracy: 0.8011
ROC-AUC: 0.5000
PR-AUC: 0.8011
MCC: 0.0000
Note: Same protein for all samples, so predictions may be constant.


In [19]:
# 3. Concatenation model
print("\n--- Concatenation Model ---")
concat_model = torch.nn.Sequential(torch.nn.Linear(DNA_EMB_DIM + PROT_EMB_DIM, 2))
concat_model.load_state_dict(torch.load(f"{RESULTS_DIR}/concat_model.pth", map_location=device))
concat_model = concat_model.to(device).eval()

with torch.no_grad():
    concat_input = torch.cat((dna_embs_tensor, protein_embs_tensor), dim=1).to(device)
    concat_out = concat_model(concat_input)
    concat_probs = torch.softmax(concat_out, dim=1)[:, 1].cpu().numpy()
    concat_preds = torch.argmax(concat_out, dim=1).cpu().numpy()

results['Concatenation'] = {
    'probs': concat_probs,
    'preds': concat_preds,
    'acc': accuracy_score(labels.numpy(), concat_preds),
    'roc_auc': roc_auc_score(labels.numpy(), concat_probs),
    'pr_auc': average_precision_score(labels.numpy(), concat_probs),
    'mcc': matthews_corrcoef(labels.numpy(), concat_preds)
}
print(f"Accuracy: {results['Concatenation']['acc']:.4f}")
print(f"ROC-AUC: {results['Concatenation']['roc_auc']:.4f}")
print(f"PR-AUC: {results['Concatenation']['pr_auc']:.4f}")
print(f"MCC: {results['Concatenation']['mcc']:.4f}")


--- Concatenation Model ---
Accuracy: 0.8011
ROC-AUC: 0.5262
PR-AUC: 0.8364
MCC: 0.0809


In [20]:
# 4. Contrastive model
print("\n--- Contrastive Model ---")
contrastive_model = CLModel()
contrastive_model.load_state_dict(torch.load(f"{RESULTS_DIR}/contrastive_model.pth", map_location=device))
contrastive_model = contrastive_model.to(device).eval()

with torch.no_grad():
    dna_proj, prot_proj = contrastive_model(dna_embs_tensor.to(device), protein_embs_tensor.to(device))
    similarities = F.cosine_similarity(dna_proj, prot_proj, dim=1)
    contr_probs = ((similarities + 1) / 2).cpu().numpy()  # Map [-1, 1] to [0, 1]
    contr_preds = (similarities > 0).long().cpu().numpy()

results['Contrastive'] = {
    'probs': contr_probs,
    'preds': contr_preds,
    'similarity': similarities.cpu().numpy(),
    'acc': accuracy_score(labels.numpy(), contr_preds),
    'roc_auc': roc_auc_score(labels.numpy(), contr_probs),
    'pr_auc': average_precision_score(labels.numpy(), contr_probs),
    'mcc': matthews_corrcoef(labels.numpy(), contr_preds)
}
print(f"Accuracy: {results['Contrastive']['acc']:.4f}")
print(f"ROC-AUC: {results['Contrastive']['roc_auc']:.4f}")
print(f"PR-AUC: {results['Contrastive']['pr_auc']:.4f}")
print(f"MCC: {results['Contrastive']['mcc']:.4f}")


--- Contrastive Model ---
Accuracy: 0.7841
ROC-AUC: 0.5945
PR-AUC: 0.8638
MCC: -0.0656


In [21]:
# 5. Attention model
print("\n--- Attention Model ---")
attention_model = AttentionModel(dna_lm, prot_lm, dna_tokenizer, prot_tokenizer)
attention_model.load_state_dict(torch.load(f"{RESULTS_DIR}/attention_model.pth", map_location=device))
attention_model = attention_model.to(device).eval()

# Process in batches to avoid memory issues
batch_size = 16
attn_probs_list = []
attn_preds_list = []

with torch.no_grad():
    for i in tqdm(range(0, len(df), batch_size), desc="Attention"):
        end_idx = min(i + batch_size, len(df))
        dna_batch = {k: v[i:end_idx].to(device) for k, v in dna_tokens.items()}
        prot_batch = {k: v[i:end_idx].to(device) for k, v in prot_tokens.items()}
        
        out, _ = attention_model(dna_batch, prot_batch)
        probs = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
        preds = torch.argmax(out, dim=1).cpu().numpy()
        attn_probs_list.extend(probs)
        attn_preds_list.extend(preds)

attn_probs = np.array(attn_probs_list)
attn_preds = np.array(attn_preds_list)

results['Attention'] = {
    'probs': attn_probs,
    'preds': attn_preds,
    'acc': accuracy_score(labels.numpy(), attn_preds),
    'roc_auc': roc_auc_score(labels.numpy(), attn_probs),
    'pr_auc': average_precision_score(labels.numpy(), attn_probs),
    'mcc': matthews_corrcoef(labels.numpy(), attn_preds)
}
print(f"Accuracy: {results['Attention']['acc']:.4f}")
print(f"ROC-AUC: {results['Attention']['roc_auc']:.4f}")
print(f"PR-AUC: {results['Attention']['pr_auc']:.4f}")
print(f"MCC: {results['Attention']['mcc']:.4f}")


--- Attention Model ---


Attention: 100%|████████████████████████████████████████████████████| 11/11 [00:00<00:00, 47.45it/s]

Accuracy: 0.7955
ROC-AUC: 0.4965
PR-AUC: 0.7818
MCC: -0.0377


In [22]:
# 6. Composition model
print("\n--- Composition Model ---")
comp_model = CompositionModel(dna_lm, prot_lm, dna_tokenizer, prot_tokenizer)
comp_model.load_state_dict(torch.load(f"{RESULTS_DIR}/comp_model.pth", map_location=device))
comp_model = comp_model.to(device).eval()

# Process in batches
comp_probs_list = []
comp_preds_list = []

with torch.no_grad():
    for i in tqdm(range(0, len(df), batch_size), desc="Composition"):
        end_idx = min(i + batch_size, len(df))
        dna_batch = {k: v[i:end_idx].to(device) for k, v in dna_tokens.items()}
        prot_batch = {k: v[i:end_idx].to(device) for k, v in prot_tokens.items()}
        
        out, _ = comp_model(dna_batch, prot_batch)
        probs = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
        preds = torch.argmax(out, dim=1).cpu().numpy()
        comp_probs_list.extend(probs)
        comp_preds_list.extend(preds)

comp_probs = np.array(comp_probs_list)
comp_preds = np.array(comp_preds_list)

results['Composition'] = {
    'probs': comp_probs,
    'preds': comp_preds,
    'acc': accuracy_score(labels.numpy(), comp_preds),
    'roc_auc': roc_auc_score(labels.numpy(), comp_probs),
    'pr_auc': average_precision_score(labels.numpy(), comp_probs),
    'mcc': matthews_corrcoef(labels.numpy(), comp_preds)
}
print(f"Accuracy: {results['Composition']['acc']:.4f}")
print(f"ROC-AUC: {results['Composition']['roc_auc']:.4f}")
print(f"PR-AUC: {results['Composition']['pr_auc']:.4f}")
print(f"MCC: {results['Composition']['mcc']:.4f}")


--- Composition Model ---


Composition: 100%|██████████████████████████████████████████████████| 11/11 [00:00<00:00, 48.37it/s]

Accuracy: 0.1989
ROC-AUC: 0.5607
PR-AUC: 0.8404
MCC: 0.0000


## 6. Summary Results (Binary Classification)

In [ ]:
# Create summary table
summary_rows = []
for model_name in ['DNA-only', 'Protein-only', 'Concatenation', 'Contrastive', 'Attention', 'Composition']:
    r = results[model_name]
    summary_rows.append({
        'Model': model_name,
        'Accuracy': f"{r['acc']:.4f}",
        'ROC-AUC': f"{r['roc_auc']:.4f}",
        'PR-AUC': f"{r['pr_auc']:.4f}",
        'MCC': f"{r['mcc']:.4f}"
    })

summary_df = pd.DataFrame(summary_rows)
print("="*60)
print("WRI1 Benchmark Results (Binary Classification: Binding vs Non-binding)")
print("="*60)
print(summary_df.to_string(index=False))

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

model_names = ['DNA-only', 'Protein-only', 'Concatenation', 'Contrastive', 'Attention', 'Composition']
metrics = ['acc', 'roc_auc', 'pr_auc', 'mcc']
metric_labels = ['Accuracy', 'ROC-AUC', 'PR-AUC', 'MCC']
colors = plt.cm.Set2(np.linspace(0, 1, len(model_names)))

for ax, metric, label in zip(axes, metrics, metric_labels):
    values = [results[m][metric] for m in model_names]
    bars = ax.bar(range(len(model_names)), values, color=colors)
    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels(model_names, rotation=45, ha='right')
    ax.set_ylabel(label)
    ax.set_title(label)
    ax.set_ylim(0, 1.1)
    
    # Add value labels on bars
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('WRI1 Benchmark: Binary Classification Performance', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/wri1_benchmark_binary.png", dpi=150, bbox_inches='tight')
plt.show()

## 7. Correlation with Binding Affinity (Kd)

For samples with measured Kd values, analyze correlation between model predictions and actual binding affinity.

In [ ]:
# Filter to samples with Kd values only
binding_mask = df['is_binding'] == 1
kd_values = df.loc[binding_mask, 'kd_numeric'].values
log_kd = np.log10(kd_values + 1e-6)  # Add small value to avoid log(0)

print(f"Analyzing {len(kd_values)} binding samples with Kd values")
print(f"Kd range: {kd_values.min():.2f} - {kd_values.max():.2f} nM")

# Calculate correlations for each model's predictions
correlation_results = []
for model_name in ['DNA-only', 'Protein-only', 'Concatenation', 'Contrastive', 'Attention', 'Composition']:
    probs = results[model_name]['probs'][binding_mask]
    
    # Spearman correlation (rank-based, better for non-linear relationships)
    spearman_corr, spearman_p = spearmanr(probs, kd_values)
    # Pearson correlation with log(Kd)
    pearson_corr, pearson_p = pearsonr(probs, log_kd)
    
    correlation_results.append({
        'Model': model_name,
        'Spearman (Kd)': f"{spearman_corr:.4f}",
        'Spearman p-val': f"{spearman_p:.4e}",
        'Pearson (log Kd)': f"{pearson_corr:.4f}",
        'Pearson p-val': f"{pearson_p:.4e}"
    })

corr_df = pd.DataFrame(correlation_results)
print("\n" + "="*60)
print("Correlation: Predicted Binding Probability vs Actual Kd")
print("Note: Higher Kd = weaker binding, so negative correlation is expected")
print("="*60)
print(corr_df.to_string(index=False))

In [ ]:
# Scatter plots: Prediction vs log(Kd)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for ax, model_name in zip(axes, ['DNA-only', 'Protein-only', 'Concatenation', 'Contrastive', 'Attention', 'Composition']):
    probs = results[model_name]['probs'][binding_mask]
    
    ax.scatter(probs, log_kd, alpha=0.5, s=20)
    ax.set_xlabel('Predicted Binding Probability')
    ax.set_ylabel('log10(Kd [nM])')
    ax.set_title(model_name)
    
    # Add correlation annotation
    spearman_corr, _ = spearmanr(probs, log_kd)
    ax.annotate(f'Spearman r = {spearman_corr:.3f}', 
                xy=(0.05, 0.95), xycoords='axes fraction',
                fontsize=10, verticalalignment='top')
    
    # Add trend line
    if len(np.unique(probs)) > 1:  # Only if there's variation
        z = np.polyfit(probs, log_kd, 1)
        p = np.poly1d(z)
        x_line = np.linspace(probs.min(), probs.max(), 100)
        ax.plot(x_line, p(x_line), "r--", alpha=0.8, linewidth=2)

plt.suptitle('Predicted Binding Probability vs Actual Binding Affinity (log Kd)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/wri1_benchmark_correlation.png", dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Results

In [ ]:
# Save summary results
summary_df.to_csv(f"{RESULTS_DIR}/wri1_benchmark_results.csv", index=False)
corr_df.to_csv(f"{RESULTS_DIR}/wri1_benchmark_correlation.csv", index=False)

# Save detailed predictions
predictions_df = df.copy()
for model_name in ['DNA-only', 'Protein-only', 'Concatenation', 'Contrastive', 'Attention', 'Composition']:
    predictions_df[f'{model_name}_prob'] = results[model_name]['probs']
    predictions_df[f'{model_name}_pred'] = results[model_name]['preds']

predictions_df.to_csv(f"{RESULTS_DIR}/wri1_benchmark_predictions.csv", index=False)

print("Results saved:")
print(f"  - {RESULTS_DIR}/wri1_benchmark_results.csv")
print(f"  - {RESULTS_DIR}/wri1_benchmark_correlation.csv")
print(f"  - {RESULTS_DIR}/wri1_benchmark_predictions.csv")
print(f"  - {RESULTS_DIR}/wri1_benchmark_binary.png")
print(f"  - {RESULTS_DIR}/wri1_benchmark_correlation.png")